# Unified Dual-Engine Systolic Matrix Accelerator (Option 2)
## Hardware Accelerated GEMM on PYNQ-Z2 (`xc7z020clg400-1`)

### Architectural Highlights:
- **512 Physical MACs**: Dual 16×16 systolic arrays operating concurrently in lockstep.
- **Plan B (Symmetrical DSP Allocation)**: Exactly 110 DSP48E1 slices per engine (220 DSPs = 100% device capacity).
- **Plan C (Unified BRAM Feeder)**: Single AXI DMA MM2S stream driving True Dual-Port BRAM to feed both engines simultaneously.
- **Plan A (BRAM Memory Mapping)**: Dedicated Block RAM primitives for tile ping-pong banks and result buffers.

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate

# Load Option 2 Unified Dual-Engine Overlay
ol = Overlay('adaptive_gemm.bit')
dma = ol.axi_dma_0
accel = ol.accel_engine
print("Overlay loaded successfully!")
print("IP Blocks Available:", [k for k in ol.ip_dict.keys()])

### 1. Reset and Configure Accelerator

In [ ]:
# Soft reset
accel.write(0x00, 0x02) # Soft reset
time.sleep(0.001)
accel.write(0x00, 0x00) # Release reset

# Configure Tile Dimensions (M=16, K=16, N=16) and Active Masks
accel.write(0x08, 16) # M
accel.write(0x0C, 16) # K
accel.write(0x10, 16) # N
accel.write(0x14, 16) # Active M rows
accel.write(0x18, 16) # Active N cols
accel.write(0x24, 0x00) # GEMM mode
print("Accelerator configured for 16x16x16 dual GEMM.")

### 2. Prepare Test Data and Allocate Contiguous DMA Memory

In [ ]:
np.random.seed(42)
A0 = np.random.randint(-128, 127, size=(16, 16), dtype=np.int8)
B0 = np.random.randint(-128, 127, size=(16, 16), dtype=np.int8)
A1 = np.random.randint(-128, 127, size=(16, 16), dtype=np.int8)
B1 = np.random.randint(-128, 127, size=(16, 16), dtype=np.int8)

# Golden reference computation
C0_golden = np.matmul(A0.astype(np.int32), B0.astype(np.int32))
C1_golden = np.matmul(A1.astype(np.int32), B1.astype(np.int32))

# Allocate contiguous DMA buffers
in_buf = allocate(shape=(256,), dtype=np.uint32)
out_buf = allocate(shape=(512,), dtype=np.int32)

# Pack A0, B0, A1, B1 into unified stream
raw_bytes = bytearray()
raw_bytes.extend(A0.tobytes())
raw_bytes.extend(B0.tobytes())
raw_bytes.extend(A1.tobytes())
raw_bytes.extend(B1.tobytes())
in_buf[:] = np.frombuffer(raw_bytes, dtype=np.uint32)[:]

in_buf.flush()
out_buf.flush()
print("Contiguous DMA buffers prepared and flushed.")

### 3. Run Hardware DMA Execution & Validate Results

In [ ]:
# Start receive channel (512 words: C0 then C1)
dma.recvchannel.transfer(out_buf)

# Start send channel (256 words: A0, B0, A1, B1)
t_start = time.time()
dma.sendchannel.transfer(in_buf)

dma.sendchannel.wait()
dma.recvchannel.wait()
t_end = time.time()

out_buf.invalidate()
C0_hw = out_buf[:256].reshape((16, 16)).copy()
C1_hw = out_buf[256:].reshape((16, 16)).copy()

# Check Bit-Exact Equivalence
np.testing.assert_array_equal(C0_hw, C0_golden)
np.testing.assert_array_equal(C1_hw, C1_golden)

hw_cycles = accel.read(0x1C)
hw_tiles = accel.read(0x20)

print("*** VERIFICATION PASSED: Bit-Exact Match with Golden NumPy! ***")
print(f"Execution Time: {(t_end - t_start)*1000.0:.3f} ms")
print(f"Hardware Performance: {hw_cycles} Clock Cycles, {hw_tiles} Tiles Processed")

### 4. Cleanup Buffers

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("DMA buffers cleanly freed.")